In [ ]:
import igl
import meshplot as mp
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
from typing import Tuple


def vec2(x: float, y: float) -> np.ndarray:
    return np.array([x, y], dtype=np.float64)


def hat(p: np.ndarray) -> np.ndarray:
    return vec2(-p[1], p[0])

def compute_triangle_areas(V: np.ndarray, T: np.ndarray) -> np.ndarray:
    E = len(T) # no of triangles/elements
    A = np.zeros(E, dtype=np.float64)
    for e in range(E):
        # indecies of vertecies of current triangle
        i, j, k = T[e, :]   
        # coordinates of vertices
        xi, yi = V[i, :]
        xj, yj = V[j, :]
        xk, yk = V[k, :]
        # distances
        dx1 = xk - xj
        dy1 = yk - yj
        dx2 = xi - xj
        dy2 = yi - yj
        # area of current element
        A[e] =  (dx1*dy2 - dx2*dy1 ) / 2.0
    return A


def make_beam_mesh(width: int, height: int, shape: Tuple[int, int]) -> Tuple[np.ndarray, np.ndarray]:
    # mesh is centered around (0, 0)
    x0 = -width/2.0
    y0 = -height/2.0
    I, J = shape #resolution
    # distances between cells
    dx = width/float(I)
    dy = height/float(J)

    V = np.zeros(((I + 1)*(J + 1), 2), np.float64)
    # vecrorized double loop
    iy, ix = np.meshgrid(np.arange(J + 1), np.arange(I + 1), indexing='ij')
    V[:, 0] = x0 + ix.ravel()*dx
    V[:, 1] = y0 + iy.ravel()*dy
    
    T = np.zeros((2*I*J,3),dtype=np.int32)
    for j in range(J):
        for i in range(I):
            k00 = (i  ) + (j  )*(I+1)
            k01 = (i+1) + (j  )*(I+1)
            k10 = (i  ) + (j+1)*(I+1)
            k11 = (i+1) + (j+1)*(I+1)
            e = 2*(i + j*I)
            if (i+j+1)%2:
                T[e,  :] = (k00,k01,k11)
                T[e+1,:] = (k00,k11,k10)
            else:
                T[e,  :] = (k10,k00,k01)
                T[e+1,:] = (k10,k01,k11)  
    return V, T

V, T = make_beam_mesh(1.0,1.0,(2,2))

class MeshInfo:
    def __init__(self, V, T):
        self.A = compute_triangle_areas(V, T)
        if np.any(np.where(self.A<=0)):
            raise ValueError('Triangle with negative orientation exist.')
        self.bc = igl.barycenter(V, T)
        # each row corresponds to a triangle
        # each entry in the row gives the index of the neighboring triangle that shares corresponding edge
        # if edge is on the boundary (i.e. no neighbpur) the entry is set to -1
        self.TT, _ = igl.triangle_triangle_adjacency(T)
        
        self.VT, self.NI = igl.vertex_triangle_adjacency(T, len(V))
